# Convert Book Images to Markdown with Google Vertex AI

This notebook uses Google Vertex AI's Gemini models to convert book page images (with equations) into markdown text.

## Setup and Installation

First, install required packages:

In [36]:
# Dependencies are pinned in requirements.txt.
# Run this notebook with the Vertex AI OCR kernel (see README.md).

## Authentication

You'll need to authenticate with Google Cloud. Options:
1. Set `GOOGLE_APPLICATION_CREDENTIALS` environment variable to point to your service account key JSON
2. Use `gcloud auth application-default login`
3. Use service account key directly

**Note**: Make sure to enable the **Vertex AI API** and **Gemini API** in your Google Cloud Console for your project before running the code.

In [37]:
# import os
# os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = 'my-service-account-key.json'

In [38]:
# Load the .env file
# Include: GOOGLE_APPLICATION_CREDENTIALS=my-service-account-key.json
from dotenv import load_dotenv
load_dotenv()

True

In [39]:
import os
from google import genai
from google.genai import types
from pathlib import Path

# Configure your Google Cloud project
PROJECT_ID = os.getenv('PROJECT_ID')
LOCATION = os.getenv('LOCATION', 'global')

# Initialize the client with Vertex AI
client = genai.Client(
    enterprise=True,
    project=PROJECT_ID,
    location=LOCATION
)

## Define the Conversion Function

In [ ]:
def image_to_markdown(image_path: str, model_name: str = "gemini-3.8-flash") -> str:
    """
    Convert a book page image to markdown using Vertex AI.
    
    Args:
        image_path: Path to the PNG image file
        model_name: Vertex AI model to use
    
    Returns:
        Markdown text extracted from the image
    """
    # Read the image file
    with open(image_path, 'rb') as f:
        image_data = f.read()
    
    # Craft a detailed prompt for OCR with mathematical content
    prompt = """Convert this book page image into markdown format. Follow these rules:

- The punctuations are ". " (period + space) and ", " (comma + space) instead of "、" and "。". Make sure to add space after these punctuations.
- Preserve the exact text and structure from the page
- Convert all mathematical equations and formulas to LaTeX notation, enclosed in:
   - Inline math: $equation$
   - Display/block math:
   $$
   display style equation
   $$
   You can also use environments like align*, equation*, etc. Insert line breaks within the LaTeX codes for readability. Add line break before and after the LaTeX block.
   For multiple equations, use align*.
- Preserve any numbered equations, theorems, or proofs. Always include the equation number within the LaTeX block with tag{}, if the equation number is present.
- ADD A LINE BREAK BEFORE AND AFTER the markdown, HTML block, and display math block $$ $$ to separate it from the following text for proper rendering.
- Maintain headings, paragraphs, lists, and formatting (including bolds, italics, superscript, subscript). The heading level follows the following rules: chapter title ##, 2 level heading (e.g. 1.1) ###, 3 level heading (e.g. 2.1.3) ####, 4 level heading (e.g. 6.1.1.3) ##### etc. It's basically heading level + 1. For big "Chapter N" texts, merge it with the chapter title that follows it (e.g. "Chapter 5: 統計数理基礎"). Use the original notation and language.
- If there are figures or diagrams, use the following format.Figure numbers and descriptions should also be stated outside the image (e.g. **図1.1** 三角関数の例). The order of image and caption should follow the original. Make sure to ADD A LINE BREAK before and after the figure div. Example:
<div class="figure">

![description](placeholder)

<div class="figure-caption">

**図1.1** 三角関数の例

</div>
</div>
- Keep the original layout structure as much as possible, but ignore the line breaks inside a sentence.
- Ignore header which includes small text e.g. "数理統計学への誘い (第5版)"
- For Japanese text, the paragraphs start with an indentation (one full-width spaces "　"). Retain this formatting. Make sure to include the indentation in the markdown output.
- Wrap theorem, definition, lemma, proof, etc. blocks with <div class="{theorem|definition|lemma|proof|proposition|axiom|example|algorithm|supplement|etc}">...</div> tags. For these block headings, use div.{blocktype}-heading. The id (e.g. 定理 1.1, アルゴリズム 2.3) would be wrapped inside span.{blocktype}-id, and its name (e.g. ピタゴラスの定理) would be tagged span.{blocktype}-name. Make sure to ADD A LINE BREAK before and after the div tag. Example:
<div class="theorem">
<div class="theorem-heading">
    <span class="theorem-id">定理1.1</span>
    <span class="theorem-name">ピタゴラスの定理</span>
</div>

$$a^2 + b^2 = c^2 \tag{1.1}$$

</div>
- No ```markdown code block.

Output only the markdown text, without any preamble or explanation.
"""
    
    # Generate content using the new google-genai API
    response = client.models.generate_content(
        model=model_name,
        contents=[
            prompt,
            types.Part.from_bytes(
                data=image_data,
                mime_type='image/png'
            )
        ]
    )
    
    return response.text

## Process a Single Image

In [35]:
# Test with a single image
folder = 'mybook/images'
filename='p095'
test_image_path = f'{folder}/{filename}.png'  # Replace with your image path
test_image_path

'mybook/images/p095.png'

In [ ]:
markdown_output = image_to_markdown(test_image_path)
print(markdown_output)

## Batch Process Multiple Pages

In [26]:
import glob
from tqdm import tqdm
import time

def process_book_pages(input_folder: str, output_file: str, model_name: str = "gemini-3.8-flash"):
    """
    Process all PNG images in a folder and combine into a single markdown file.
    
    Args:
        input_folder: Folder containing PNG page images
        output_file: Path to save the combined markdown output
        model_name: Vertex AI model to use
    """
    # Get all PNG files sorted by name
    image_files = sorted(glob.glob(os.path.join(input_folder, "*.png")))
    
    print(f"Found {len(image_files)} images to process")
    
    all_markdown = []
    processed_results = []  # Store results as we process
    
    for i, image_path in enumerate(tqdm(image_files), 1):
        try:
            # print(f"\nProcessing page {i}: {os.path.basename(image_path)}")
            markdown = image_to_markdown(image_path, model_name)
            
            # Store the result immediately
            page_content = f"{markdown}"
            processed_results.append(page_content)
            
            # Rate limiting: pause briefly between requests
            time.sleep(1)
            
        except Exception as e:
            print(f"Error processing {image_path}: {str(e)}")
            error_content = f"\nError: {str(e)}\n\n"
            processed_results.append(error_content)
    
    # Concatenate all results at the end
    all_markdown = ''.join(processed_results)
    
    # Save combined markdown
    with open(output_file, 'w', encoding='utf-8') as f:
        f.write(all_markdown)
    
    print(f"\nComplete. Saved to {output_file}")
    
    # Return the results so you can access them even if interrupted
    return processed_results

In [ ]:
# Process all pages in a folder
folder = 'path/to/images/chapter0'
INPUT_FOLDER = folder
OUTPUT_FILE = "ch0.md"

process_book_pages(INPUT_FOLDER, OUTPUT_FILE)